# Robot Chasing — always up baseline

Predict absolute action `0=up` for every observation.

In [7]:
import json, os
from pathlib import Path
# assumes that the dataset https://huggingface.co/datasets/IOAI-official/ioai-2026-robot-chasing is fully downloaded
# using git clone and then git lfs pull
# into the current directory and the dataset directory is already renamed 'dataset'
root = Path(os.environ.get("DATASET_ROOT", "dataset"))
observations = json.load(open(root / "private/test_leaderboard_b" / "observations.json"))
'''
# Absolute actions: 0=up, 1=down, 2=left, 3=right, 4=pickup, 5=drop.
predictions = [0 for _ in observations]

with open("predictions.json", "w") as handle:
    json.dump(predictions, handle)
print("wrote", len(predictions), "predictions to predictions.json")'''

wrote 3600 predictions to predictions.json


In [32]:
import json
from tqdm import tqdm
import numpy as np
y = json.load(open('dataset/public/train_answers.json'))
X = json.load(open('dataset/public/train/observations.json'))
COLOUR_NAMES = {
    0: "red",
    1: "green",
    2: "blue",
    3: "purple",
    4: "yellow",
    5: "grey",
}
OBJECT_NAMES = {5: "key", 6: "ball", 7: "box", 11: "token"}
cn = {v: k for k, v in COLOUR_NAMES.items()}
on = {v: k for k, v in OBJECT_NAMES.items()}
def extract_features(X):
    ftrs = []
    for item in tqdm(X):
        mission = item['mission']
        robot_loc = [int(x[0]) for x in list(np.where(np.asarray(item['image'])[:,:,0] == 10))]
        c, o = None, None
        for q in mission.split(' '):
            if q in cn.keys():
                c = cn[q]
                break
        for q in mission.split(' '):
            if q in on.keys():
                o = on[q]
                break
        match = (np.asarray(item['image'])[:,:,0] == o).astype('int') * (np.asarray(item['image'])[:,:,1] == c).astype('int')
        if match.sum() == 0:
            feature = [item['robot_id'], item['direction'], 0.0, 0.0]
        else:
            matches = np.array(np.where(match == 1)).T
            closest, record = matches[0], float('inf')
            for a in matches:
                dist = (a[0] - robot_loc[0]) ** 2 + (a[1] - robot_loc[1]) ** 2
                if dist < record:
                    record = dist
                    closest = a
            # use the closest target
            feature = [item['robot_id'], item['direction'], closest[0] - robot_loc[0], closest[1] - robot_loc[1]]
        ftrs.append(np.concatenate([np.array(feature), np.asarray(item['image'])[robot_loc[0]-1:robot_loc[0]+2,robot_loc[1]-1:robot_loc[1]+2].flatten()]))
        #ftrs.append(np.concatenate([np.asarray(item['image']).flatten(), np.array(item['direction']).reshape(-1)]))
        # tip: instead of feeding the whole image into xgboost, feed the image closest to the area of the robot
    return ftrs
ftrs = extract_features(X)
ftrs = np.array(ftrs)
y = np.array(y)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
trees = []
for i in range(6):
    xt, xv, yt, yv = train_test_split(ftrs[ftrs[:,0] == i], y[ftrs[:,0] == i], test_size=0.1, random_state=42)
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    #cls = DecisionTreeClassifier()
    cls = XGBClassifier(reg_lambda=3.0, random_state=42) # i feel like xgb tends to overfit a bit especially when it's given the full grid
    cls.fit(xt[:,1:], yt)
    from sklearn.metrics import accuracy_score
    print(accuracy_score(yv, cls.predict(xv[:,1:])))
    cls.fit(ftrs[:,1:], y)
    trees.append(cls)
    # split modeling by robot id: 43% local validation acc
    # 52.3% local validation with xgboost w/ default hyperparameters

100%|██████████| 60000/60000 [00:03<00:00, 18315.27it/s]


0.549
0.513
0.529
0.545
0.472
0.534


In [34]:
observations = json.load(open(root / "private/test_leaderboard_b" / "observations.json"))
test_ftrs = extract_features(observations)
preds = []
from tqdm import tqdm
for item in tqdm(test_ftrs):
    classifier = trees[int(item[0])]
    arr = np.array(item[1:]).reshape(1, -1)
    preds.append(int(classifier.predict(arr)[0]))
import json, os
from pathlib import Path

root = Path(os.environ.get("DATASET_ROOT", "dataset"))

# Absolute actions: 0=up, 1=down, 2=left, 3=right, 4=pickup, 5=drop.

with open("predictions.json", "w") as handle:
    json.dump(preds, handle)
print("wrote", len(predictions), "predictions to predictions.json")

100%|██████████| 3600/3600 [00:06<00:00, 519.81it/s]

wrote 3600 predictions to predictions.json


In [35]:
ans = 'dataset/private/test_leaderboard_b_answers.json'
lb_answers = json.load(open(ans))
import numpy as np
(np.array(preds) == np.array(lb_answers)).sum() / len(lb_answers)
# leaderboard B score 0.5205 :)

np.float64(0.5205555555555555)